# 02 · Calidad de datos

**Objetivo:** revisar el reporte de calidad que genera el pipeline (paso 5) y entender los
problemas conocidos de la fuente antes de analizar.

**Entradas:** `data/processed/reporte_calidad.csv` y `data/processed/perfil_columnas.csv`.

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"      # capa gold
PROCESSED = ROOT / "data" / "processed"  # capa silver
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
millones = mticker.FuncFormatter(lambda x, _: f"{x / 1e6:,.0f} M")
print("Proyecto:", ROOT)

## 1. Resultado de las reglas

In [ ]:
calidad = pd.read_csv(PROCESSED / "reporte_calidad.csv")
print(calidad.groupby(["severidad", "estado"]).size().unstack(fill_value=0))
calidad[["tabla", "regla", "severidad", "estado", "filas_evaluadas", "filas_con_problema",
         "pct_problema", "umbral_pct"]]

In [ ]:
# Reglas que fallaron (superan su umbral)
calidad[calidad["estado"] == "FALLA"][["tabla", "regla", "severidad", "pct_problema", "detalle"]]

## 2. Problemas conocidos de la fuente

El Registro de Datos de OCP documenta problemas de calidad en la publicación del OECE. El pipeline
los detecta y los reporta como reglas **INFO**:

In [ ]:
calidad[calidad["severidad"] == "INFO"][["tabla", "regla", "filas_con_problema",
                                           "pct_problema", "detalle"]]

## 3. Perfil de columnas (nulos y cardinalidad)

In [ ]:
perfil = pd.read_csv(PROCESSED / "perfil_columnas.csv")
nulos = perfil[perfil["pct_nulos"] > 0].sort_values("pct_nulos", ascending=True).tail(15)
etiquetas = nulos["tabla"] + "." + nulos["columna"]
plt.barh(etiquetas, nulos["pct_nulos"], color=PALETA[1])
plt.title("Columnas con más valores nulos (%)"); plt.xlabel("% de nulos")
plt.show()

In [ ]:
perfil[perfil["tabla"] == "procesos"]

## Conclusiones

* Una regla **ERROR** fallida invalida el análisis (configurable con `quality.fail_on_error`).
* Las **ADVERTENCIAS** reducen la cobertura: hay que mencionarlas al presentar resultados.
* Las reglas **INFO** documentan características conocidas de la fuente que el pipeline ya maneja.
* Siguiente notebook: **03 · Análisis del gasto**.